# 06. 전이학습 (Transfer Learning)

ImageNet에서 사전학습된 ResNet18을 가져와 CIFAR-10(10개 클래스, 32×32 컬러 이미지)에 맞게 미세조정합니다.

**학습 목표**
- `torchvision.models`에서 사전학습 가중치를 불러온다
- backbone을 고정(freeze)하고 마지막 분류층만 새로 학습한다
- 사전학습 모델이 요구하는 전처리(크기, 정규화)를 맞춘다

데이터: `data/downloads/` (CIFAR-10과 사전학습 가중치, 첫 실행 시 자동 다운로드, git 제외)

> CPU에서도 빨리 끝나도록 학습 이미지 2,000장, 테스트 이미지 1,000장만 사용합니다.

In [ ]:
import os
from pathlib import Path

DATA = Path("data")
# 사전학습 가중치도 섹션 폴더에 저장되도록 torch hub 캐시 위치를 지정 (torch import 전에 설정)
os.environ["TORCH_HOME"] = str((DATA / "downloads").resolve())

import torch

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__}, device: {device}")

## 1. 데이터 준비

사전학습 모델은 224×224 입력과 ImageNet 정규화 값으로 학습되었으므로 같은 전처리를 사용합니다. 학습 데이터에는 좌우 반전 증강을 추가합니다.

In [ ]:
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, Subset

weights = models.ResNet18_Weights.IMAGENET1K_V1
mean, std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

train_tf = transforms.Compose([
    transforms.Resize(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
test_tf = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

full_train = datasets.CIFAR10(DATA / "downloads", train=True, download=True, transform=train_tf)
full_test = datasets.CIFAR10(DATA / "downloads", train=False, download=True, transform=test_tf)
classes = full_train.classes

g = torch.Generator().manual_seed(42)
train_ds = Subset(full_train, torch.randperm(len(full_train), generator=g)[:2000].tolist())
test_ds = Subset(full_test, torch.randperm(len(full_test), generator=g)[:1000].tolist())

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=128, num_workers=2)
print("classes:", classes)
print(f"train {len(train_ds)} | test {len(test_ds)}")

In [ ]:
import matplotlib.pyplot as plt

def denorm(img):
    return (img * torch.tensor(std).view(3, 1, 1) + torch.tensor(mean).view(3, 1, 1)).clamp(0, 1)

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, i in zip(axes, range(8)):
    img, label = test_ds[i]
    ax.imshow(denorm(img).permute(1, 2, 0))
    ax.set_title(classes[label])
    ax.axis("off")
plt.show()

## 2. 사전학습 모델 불러오기와 레이어 고정

backbone의 파라미터는 `requires_grad=False`로 고정하고, 마지막 `fc` 층을 10개 클래스용으로 교체합니다. 새로 만든 층은 기본값이 `requires_grad=True`입니다.

In [ ]:
from torch import nn

model = models.resnet18(weights=weights)
for p in model.parameters():
    p.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, len(classes))
model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"학습 파라미터 {trainable:,} / 전체 {total:,}")

## 3. 학습과 평가

옵티마이저에는 학습할 파라미터(`model.fc`)만 넘깁니다.

In [ ]:
from torch import optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)

def evaluate():
    model.eval()
    correct = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(1) == y).sum().item()
    return correct / len(test_ds)

print(f"학습 전 test acc: {evaluate():.3f} (무작위 기준 0.100)")
for epoch in range(1, 3):
    model.train()
    total_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
    print(f"epoch {epoch} | train loss {total_loss / len(train_ds):.4f} | test acc {evaluate():.3f}")

## 4. 예측 확인

In [ ]:
x, y = next(iter(test_loader))
model.eval()
with torch.no_grad():
    pred = model(x.to(device)).argmax(1).cpu()

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for ax, i in zip(axes.flat, range(16)):
    ax.imshow(denorm(x[i]).permute(1, 2, 0))
    ax.set_title(classes[pred[i]], color="green" if pred[i] == y[i] else "red", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 요약
- `models.resnet18(weights=...)`로 사전학습 모델을 불러온다
- backbone을 freeze하고 `fc`만 교체해 학습하면, 적은 데이터와 짧은 학습으로도 높은 정확도를 얻는다
- 사전학습 때와 같은 입력 크기와 정규화를 사용해야 한다
- 성능을 더 올리려면 backbone 일부를 풀고(unfreeze) 작은 학습률로 함께 미세조정한다